In [ ]:
!pip install torch torchvision elpv-dataset scikit-learn opencv-python

In [ ]:
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from elpv_dataset.utils import load_dataset

# 1. Carregamento e Preparação dos Dados
print("A carregar o dataset ELPV...")
images, proba, types = load_dataset()
print(f"Total de imagens carregadas: {images.shape[0]}")

# Ajuste de canais e dimensões para o PyTorch (N, C, H, W)
X = np.stack([images, images, images], axis=-1)
X = X.transpose(0, 3, 1, 2).astype(np.float32)
y = proba.astype(np.float32)

# Divisão de Dados (70% Treino, 15% Validação, 15% Teste)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)


# 2. Dataset Customizado com Data Augmentation leve para treino
class ELPVDataset(Dataset):

  def __init__(self, images, labels, augment=False):
    self.images = torch.as_tensor(images, dtype=torch.float32)
    if self.images.max() > 1.0:
      self.images = self.images / 255.0
    self.labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)
    self.augment = augment

    # Transformações geométricas leves para robustez
    self.transform = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=20),
    ])

  def __len__(self):
    return len(self.images)

  def __getitem__(self, idx):
    img = self.images[idx]
    if self.augment:
      img = self.transform(img)
    return img, self.labels[idx]


# Hiperparâmetros
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.0001

train_loader = DataLoader(
    ELPVDataset(X_train, y_train, augment=True),
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=True,
)
val_loader = DataLoader(
    ELPVDataset(X_val, y_val, augment=False),
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=True,
)
test_loader = DataLoader(
    ELPVDataset(X_test, y_test, augment=False),
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=True,
)

# 3. Configuração do Modelo (ResNet18 End-to-End)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo em uso: {device}")

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in model.parameters():
  param.requires_grad = True

# Substituição da camada final para Regressão Contínua [0, 1]
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_ftrs, 128),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(128, 1),
    nn.Sigmoid(),
)

model = model.to(device)
criterion = nn.MSELoss()

optimizer = optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=1e-2
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

# 4. Loop de Treinamento Otimizado
best_val_loss = float("inf")
caminho_modelo = os.path.join('../models', 'resnet18_elpv_test.pth')

print("\nInício do treino otimizado com ResNet18...")
for epoch in range(EPOCHS):
  model.train()
  running_loss = 0.0
  for inputs, targets in train_loader:
    inputs, targets = inputs.to(device), targets.to(device)

    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, targets)
    loss.backward()
    optimizer.step()

    running_loss += loss.item() * inputs.size(0)

  epoch_train_loss = running_loss / len(train_loader.dataset)

  # Validação
  model.eval()
  val_loss = 0.0
  with torch.no_grad():
    for inputs, targets in val_loader:
      inputs, targets = inputs.to(device), targets.to(device)
      outputs = model(inputs)
      loss = criterion(outputs, targets)
      val_loss += loss.item() * inputs.size(0)

  epoch_val_loss = val_loss / len(val_loader.dataset)

  scheduler.step(epoch_val_loss)

  print(
      f"Época [{epoch+1}/{EPOCHS}] - Loss Treino: {epoch_train_loss:.4f} | Loss"
      f" Val: {epoch_val_loss:.4f}"
  )

  # Guardar o melhor modelo da época
  if epoch_val_loss < best_val_loss:
    best_val_loss = epoch_val_loss
    torch.save(model.state_dict(), caminho_modelo)

print(
    "\nTreinamento concluído com sucesso! Melhores pesos guardados em"
    " {caminho_modelo}."
)

# Carregar os melhores pesos salvos para a avaliação final
model.load_state_dict(torch.load(caminho_modelo))

# 5. Avaliação Final no Conjunto de Teste
model.eval()
y_true_list, y_pred_list = [], []

with torch.no_grad():
  for inputs, targets in test_loader:
    inputs = inputs.to(device)
    outputs = model(inputs)
    y_true_list.extend(targets.cpu().numpy())
    y_pred_list.extend(outputs.cpu().numpy())

y_true = np.array(y_true_list).flatten()
y_pred = np.array(y_pred_list).flatten()

# Métricas de Regressão
mse = mean_squared_error(y_true, y_pred)
mae = mean_absolute_error(y_true, y_pred)

print(f"\n==========================================")
print(f"       RELATÓRIO DE DESEMPENHO (TESTE)    ")
print(f"==========================================")
print(f"MSE: {mse:.4f}")
print(f"MAE: {mae:.4f}")

# 5.1 Métricas de Classificação (com Limiar / Threshold = 0.5)
THRESHOLD = 0.5
y_true_binary = (y_true >= THRESHOLD).astype(int)
y_pred_binary = (y_pred >= THRESHOLD).astype(int)

accuracy = accuracy_score(y_true_binary, y_pred_binary)
precision = precision_score(y_true_binary, y_pred_binary, zero_division=0)
recall = recall_score(y_true_binary, y_pred_binary, zero_division=0)
f1 = f1_score(y_true_binary, y_pred_binary, zero_division=0)

print(f"\n--- Métricas de Classificação (Threshold = {THRESHOLD}) ---")
print(f"Acurácia  (Accuracy):  {accuracy * 100:.2f}%")
print(f"Precisão  (Precision): {precision:.4f}")
print(f"Recall    (Recall):    {recall:.4f}")
print(f"F1-Score  (F1):        {f1:.4f}")

print("\n--- Relatório de Classificação Detalhado ---")
print(classification_report(y_true_binary, y_pred_binary, zero_division=0))

print("--- Matriz de Confusão (Raw) ---")
print(confusion_matrix(y_true_binary, y_pred_binary))
print(f"==========================================")